# 08 — The background agent: drop folder -> matching -> toasts

> **Restart kernel + Run All.** After editing any `.py` file: restart the kernel,
> then Run All (no `%autoreload` — it leaves two copies of the same Pydantic class
> alive and breaks validation).

> **Cost gate:** the agent runs the SAME pipeline as notebooks 03-05 behind the
> same switches. Remote ingestion happens only in **agent mode** with
> `RUN_LIVE_LLM=1` and a real provider; **demo cycles — the 10 s loop and any
> cycle after `stop()` — never touch the network at all** (no feeds, no model
> calls). The hourly `agent` mode honors `RUN_LIVE_LLM` as configured. This
> notebook runs fully offline with `LLM_PROVIDER=fake`.

**The loop** (one "cycle"):

1. **drop folder** — a JSON or HTML file dropped into `data/live_feed/` is
   parsed on the next cycle as a new opportunity (JSON is validated as-is, HTML
   goes through the same rules-first extraction as web pages). Files are never
   deleted: parsed ones move to `consumed/`, broken ones to `quarantine/`;
2. **ingestion** — only when live (see the cost gate above);
3. **matching** — every stored profile is scored against every stored
   opportunity (notebook 05's pipeline);
4. **notification rules** — threshold, daily cap, quiet hours 22-08, digest,
   "never twice for the same opportunity" (notebook 05, step 7);
5. **toasts** — every SENT notification pops an ipywidgets toast in this
   notebook: title (link to the source), score and the top reasons.

**The loop can never crash the kernel:** every stage is wrapped; a failing
source, a broken drop file or a dead widget is recorded in the cycle report and
the next tick still comes.

> **Privacy:** the drop folder and the store are local files. Remote calls (live
> mode only) are the same ones as in notebooks 03-04; no user document is ever
> sent anywhere by this notebook.


In [ ]:
import json
import os
import sys
import time
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

from oppradar_ai.agent import BackgroundAgent, status_rows
from oppradar_ai.config import Settings, get_settings, setup_logging
from oppradar_ai.schemas import Notification, Opportunity, Profile
from oppradar_ai.store import JsonStore
from oppradar_ai.toasts import ToastBoard

# -- Cost gate: the agent calls the model only behind RUN_LIVE_LLM ------------
RUN_LIVE_LLM = Settings().run_live_llm
os.environ.pop('LLM_PROVIDER', None)      # drop the force of a previous run
_configured = Settings().llm_provider     # fresh read of ai/.env (get_settings is cached)
_can_live = _configured != 'gemini' or bool(Settings().gemini_key_value)
os.environ['LLM_PROVIDER'] = _configured if RUN_LIVE_LLM and _can_live else 'fake'
get_settings.cache_clear()

setup_logging('INFO')
settings = get_settings().ensure_dirs()
print('live LLM:', 'yes' if RUN_LIVE_LLM and _can_live else 'no',
      '| provider:', settings.llm_provider,
      '| demo tick:', settings.demo_interval_seconds, 's',
      '| agent tick:', settings.agent_interval_seconds, 's')


## Step 1 — Meet the agent: a status table you can trust

The `BackgroundAgent` owns one scheduler thread and one loop. This demo scores
against a **demo store** (`data/store_agent_demo/`, wiped on every run) seeded
with notebook 05's fixtures, so your real store stays untouched. The
`ToastBoard` is wired to the agent's SENT notifications with one line.


In [ ]:
DEMO_STORE = JsonStore(root=settings.data_dir / 'store_agent_demo')
DEMO_STORE.reset()  # a fresh demo on every run; your real store is untouched

fixtures = settings.data_dir / 'fixtures'
profiles = [Profile.model_validate(p)
            for p in json.loads((fixtures / 'profiles.json').read_text(encoding='utf-8'))]
opps = [Opportunity.model_validate(o)
        for o in json.loads((fixtures / 'opportunities.json').read_text(encoding='utf-8'))]
for profile in profiles:
    DEMO_STORE.save_user(profile)
DEMO_STORE.save_opportunities(opps)
print(f'demo store: {len(opps)} opportunities x {len(profiles)} profiles')

# sources=None -> sources.yaml is read on live cycles only (see the cost gate).
agent = BackgroundAgent(settings=settings, store=DEMO_STORE, sources=None)
board = ToastBoard(max_visible=5)
agent.add_notification_callback(board.push)  # every SENT notification -> a toast
print('agent wired to the toast board')

display(pd.DataFrame(status_rows(agent.status()), columns=['field', 'value']))


## Step 2 — The drop folder: a JSON file becomes an opportunity

Drop a `.json` or `.html` file into `data/live_feed/` (here we write one for
you) and run one cycle **by hand** with `run_cycle()`. The JSON is validated
as-is — no model involved; an HTML file would go through the same
rules-first extraction as any web page. The record lands in the store with
`added_via='user'` and the source `drop:<filename>`.

A file younger than half a second is left alone — it may still be mid-write, so
the next cycle picks it up. We just wrote ours, hence the short `sleep` below.

> **Heads-up:** a cycle consumes *every* `.json`/`.html` file sitting in
> `data/live_feed/` (into `consumed/` or `quarantine/`) — move your own files
> out before running this demo.

A manual `run_cycle()` uses **agent-mode** settings — with `RUN_LIVE_LLM=1` in
your `.env` this does **one real ingestion** of `sources.yaml` (8 feeds, capped
at `MAX_EXTRACTIONS_PER_RUN` model calls). The free tier allows 5 calls per
minute, so most items of a big run are recorded as **pending** and retried by
the next *hourly* run: a wall of `pending LLM ... quota exhausted` lines is the
quota gate working as designed, **not a crash** — the summary line at the end
tells you what was accepted, rejected and deferred. The **scheduled demo ticks
(step 4) and every cycle after `stop()` are demo mode: they never touch the
network at all.**


In [ ]:
demo_drop = {
    'title': 'Demo: Synthetic ML Internship (notebook 08)',
    'organizer': 'OppRadar Demo Org',
    'type': 'internship',
    'mode': 'remote',
    'country': 'Algeria',
    'deadline': '2027-06-30',
    'funding': '15,000 DZD/month',
    'summary': 'Synthetic record written by this notebook to demonstrate the drop folder.',
    'url': 'https://example.org/demo-drop',
}
drop_path = settings.live_feed_dir / 'demo_drop.json'
drop_path.write_text(json.dumps(demo_drop), encoding='utf-8')
time.sleep(1)  # let the file "settle": the scanner ignores files < 0.5 s old
print('dropped:', drop_path.name)

report = agent.run_cycle()
print('drop consumed:', report.drop.consumed)
print('ingestion:', report.ingestion_skipped or 'live (sources.yaml)')
print(f"items found: {report.items_found} | matches: {report.matches} | "
      f"notifications sent: {report.notifications_sent} "
      f"(pending {report.notifications_pending}, "
      f"suppressed {report.notifications_suppressed})")

amine_notes = DEMO_STORE.notification_history('u-amine', limit=10)
if amine_notes:
    display(pd.DataFrame([n.model_dump() for n in amine_notes])
            [['title', 'score', 'status', 'suppressed_reason']])
else:
    print('no notifications for u-amine this cycle')


## Step 3 — Toasts: what pops when a match passes the rules

Every **SENT** notification pops a toast: title (a link to the source), score
and the top reasons. A **PENDING** (quiet hours 22-08, daily cap, digest) or
**SUPPRESSED** (already notified) row never toasts — that is the notification
rules working, and the table above shows exactly why.

If your clock is inside quiet hours, nothing was SENT just now, so the cell
below pushes one **simulated** toast (clearly labelled) to show the widget.


In [ ]:
display(board.container)

if not board.delivered:
    top = max(DEMO_STORE.get_matches('u-amine'), key=lambda m: m.score, default=None)
    if top:
        board.push(Notification(
            profile_id='u-amine', opportunity_id=top.opportunity.id,
            title=top.opportunity.title,
            reason=top.top_reasons[0] if top.top_reasons else '',
            score=top.score, deadline=top.opportunity.deadline,
            url=top.opportunity.url,
        ), top)
        print('SIMULATED push: one toast for illustration (nothing was SENT now)')
print('toasts delivered this session:', len(board.delivered))


## Step 4 — Demo mode: the loop runs by itself (10 s ticks)

`start('demo')` schedules a cycle every `DEMO_INTERVAL_SECONDS` on a background
thread; `pause()` / `resume()` / `stop()` control it. We wait for one tick and
read the status table: `last run`, `next run`, `items found`, `matches`,
`notifications`. The second cycle re-scores everything but the rules suppress
every already-notified opportunity — no duplicate toasts. A demo tick only
looks at the local store and the drop folder; **it never fetches remote
sources**, whatever your `.env` says.

> The real setup is one word: `agent.start('agent')` (every
> `AGENT_INTERVAL_SECONDS`, i.e. 1 h) with `RUN_LIVE_LLM=1` in `ai/.env`.


In [ ]:
agent.start(mode='demo')
time.sleep(settings.demo_interval_seconds + 3)  # catch at least one tick

for label, value in status_rows(agent.status()):
    print(f'{label}: {value}')

agent.pause()
print('after pause ->', agent.status().state)
agent.resume()
print('after resume ->', agent.status().state)
agent.stop()
print('after stop ->', agent.status().state)
print('agent survived:', agent.status().state == 'stopped')


## Step 5 — Never crash: a broken file is quarantined, the loop lives

An unreadable drop file becomes an error line in the cycle report and the file
is moved to `data/live_feed/quarantine/` — never deleted, never retried in a
loop. The same belt-and-braces wrap covers sources, the store and even a toast
widget that raises.


In [ ]:
broken = settings.live_feed_dir / 'broken_drop.json'
broken.write_text('{not json at all', encoding='utf-8')
time.sleep(1)  # let the file "settle" before the scan, as in step 2

report = agent.run_cycle()  # the scheduler is stopped; this is a manual cycle
print('quarantine:', report.drop.quarantined)
print('errors:', report.errors)
print('agent survived:', agent.status().state == 'stopped')


## Recap

- **one loop**: drop folder -> ingestion (live only) -> matching -> rules ->
  toasts, on APScheduler's thread; demo = 10 s, agent = 1 h;
- **the drop folder** turns a JSON/HTML file in `data/live_feed/` into a stored
  opportunity (`added_via='user'`), consuming it into `consumed/` or
  `quarantine/` — files are never deleted;
- **toasts** show title + link, score and the top reasons for SENT rows only;
  pending/suppressed rows explain their silence in the cycle report;
- **the loop cannot crash the kernel**: every stage is wrapped and reported;
- **cost**: demo mode never spends quota; the hourly agent honors
  `RUN_LIVE_LLM` and the per-run budgets of notebooks 03-04 (plus the capped
  phrasing of notebook 05, at most `MAX_PHRASES_PER_RUN` per cycle).
